# CSE3068: Sequential and Spatial Data Mining
## Project: RescuePath AI — Intelligent Disaster Response & Evacuation Platform
**Student:** Karthikeyan A (23MIA1123)

---

### Overview
This notebook demonstrates the algorithmic foundations of **RescuePath AI**, broken down into:
1. **Sequential Data Mining**: Multi-step horizon flood risk forecasting (24h, 48h, 72h) from hydro-meteorological time-series sequences.
2. **Spatial Data Mining**: ST-DBSCAN hazard clustering, Global Moran's I spatial autocorrelation, and 2D Kernel Density Estimation (KDE).
3. **Spatial Graph Mining**: Risk-weighted A* and Dijkstra pathfinding on road network graphs for optimal evacuation routing.

In [ ]:
import os, sys
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

# Ensure backend is in python path
sys.path.append(os.path.abspath('..'))

from backend.app.database.seed_data import get_region_data
from backend.app.core.sequential_miner import sequential_miner
from backend.app.core.spatial_miner import spatial_miner
from backend.app.core.graph_router import graph_router

print("All RescuePath AI SSDM core modules loaded successfully!")

## Part 1: Sequential Data Mining
### Hydro-Meteorological Time-Series Analysis & Multi-Step Flood Prediction

In [ ]:
# Load historical sequence for Kerala Periyar River Basin
data = get_region_data('kerala_ernakulam')
sequence = data['sequence']

df_seq = pd.DataFrame(sequence)
print("Historical Sequence (Past 72 Hours in 6h intervals):")
print(df_seq[['hour_step', 'rainfall_mm', 'discharge_cumecs', 'soil_sat', 'river_level_m']])

# Extract sequential temporal features
features = sequential_miner.extract_sequential_features(sequence)
print("\nExtracted Sequential Features:")
for k, v in features.items():
    print(f" - {k}: {v:.3f}")

In [ ]:
# Execute Multi-Step Sequential Horizon Prediction (24h, 48h, 72h)
forecast = sequential_miner.predict_multi_step_horizon(sequence)

print(f"Sequence Trend: {forecast['sequence_trend']}")
print(f"Confidence: {forecast['confidence_score'] * 100:.1f}%")

for h in ['forecast_24h', 'forecast_48h', 'forecast_72h']:
    step = forecast[h]
    print(f"\nHorizon +{step['horizon_hours']}h:")
    print(f"  Flood Probability: {step['flood_probability'] * 100:.1f}%")
    print(f"  Risk Classification: {step['risk_level']}")
    print(f"  Expected River Gauge: {step['expected_river_level_m']} m (Danger Mark: 7.50 m)")
    print(f"  Driving Factors: {', '.join(step['driving_factors'])}")

## Part 2: Spatial Data Mining
### ST-DBSCAN Clustering, Moran's I Autocorrelation & 2D Gaussian KDE

In [ ]:
# Run DBSCAN over telemetry sensor points
sensors = data['sensors']
cluster_res = spatial_miner.run_dbscan_clustering(sensors, eps_km=1.8, min_samples=3)

print(f"Total Telemetry Incidents: {cluster_res['total_incidents']}")
print(f"Coherent Hazard Clusters: {cluster_res['num_clusters']}")
print(f"Noise/Outlier Points Filtered: {cluster_res['noise_points_count']}")
print(f"Global Moran's I: {cluster_res['spatial_autocorrelation_morans_i']} (p-val: {cluster_res['p_value']})")
print(f"Spatial Autocorrelation Pattern: {cluster_res['spatial_pattern']}")

for c in cluster_res['clusters']:
    print(f"Cluster #{c['cluster_id']}: {c['hazard_level']} | Centroid: {c['centroid']} | Radius: {c['radius_km']} km | Depth: {c['avg_water_depth_cm']} cm")

In [ ]:
# Compute 2D Gaussian Kernel Density Estimation (KDE) surface
kde_res = spatial_miner.compute_kde_risk_grid(sensors, resolution=25)
grid = np.array(kde_res['grid'])

print(f"KDE Grid Resolution: {grid.shape}")
print(f"Peak Risk Intensity: {np.max(grid):.2f}")

## Part 3: Spatial Graph Mining & Dynamic Evacuation Routing
### Comparative Network Analysis: Standard Shortest Path vs Risk-Penalized A* Safe Corridor

In [ ]:
# Evacuate from Aluva Manappuram (flooded riverbank)
start_lat = 10.1085
start_lng = 76.3535

routes = graph_router.compute_evacuation_routes(
    region_id='kerala_ernakulam',
    start_lat=start_lat,
    start_lng=start_lng,
    flood_surge_active=True
)

print("Target Shelter Allocated:", routes['target_shelter']['name'])
print(f"Risk Reduction Achieved: {routes['risk_reduction_percentage']:.1f}%")
print(f"Recommendation: {routes['recommendation']}")

print("\n--- Safe Route Details ---")
print(f"Distance: {routes['safe_route']['total_distance_km']} km")
print(f"Time: {routes['safe_route']['estimated_time_min']} mins")
print(f"Risk Exposure: {routes['safe_route']['risk_exposure_score']}")
print(f"Submerged Segments: {routes['safe_route']['submerged_segments_encountered']}")

print("\n--- Standard Unsafe Route Details ---")
print(f"Distance: {routes['standard_route']['total_distance_km']} km")
print(f"Time: {routes['standard_route']['estimated_time_min']} mins")
print(f"Risk Exposure: {routes['standard_route']['risk_exposure_score']}")
print(f"Submerged Segments: {routes['standard_route']['submerged_segments_encountered']}")